# Conjunto de datos completo sin clusterización

In [1]:
#Importaciones
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

#Lectura de datos
datos = pd.read_excel('03_Clusterizacion_CTNET.xlsx')
datos.head(24)

,Fecha,Generación,Temperatura,Humedad Relativa,Índice UV,Hora,Cluster KMeans,Cluster GMM
0,2022-09-01 00:00:00,0.000000,19,77,0,0,Noche,Noche
1,2022-09-01 01:00:00,0.000000,19,82,0,1,Noche,Noche
2,2022-09-01 02:00:00,0.000000,18,85,0,2,Noche,Noche
3,2022-09-01 03:00:00,0.000000,18,87,0,3,Noche,Noche
4,2022-09-01 04:00:00,0.000000,18,88,0,4,Noche,Noche
5,2022-09-01 05:00:00,0.000000,17,86,0,5,Noche,Noche
6,2022-09-01 06:00:00,0.000000,18,89,0,6,Soleado,Lluvioso
7,2022-09-01 07:00:00,6.584959,18,95,0,7,Soleado,Lluvioso
8,2022-09-01 08:00:00,560.422022,18,100,0,8,Soleado,Lluvioso
9,2022-09-01 09:00:00,7720.582326,18,100,1,9,Soleado,Lluvioso


In [2]:
datos["Generacion_prev_hour"] = datos["Generación"].shift(1)
datos["Generacion_prev_day"] = datos["Generación"].shift(24)
datos = datos.dropna(how="any", axis= 0)

Definimos X y y

In [3]:
datos_dia = datos[datos["Cluster GMM"] == "Nublado"].copy()
datos_dia.head(10)

,Fecha,Generación,Temperatura,Humedad Relativa,Índice UV,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day
36,2022-09-02 12:00:00,27523.885172,21,57,4,12,Nublado,Nublado,17036.043251,29196.986647
37,2022-09-02 13:00:00,20596.278869,23,45,5,13,Nublado,Nublado,27523.885172,25478.471342
38,2022-09-02 14:00:00,28500.000000,24,37,6,14,Nublado,Nublado,20596.278869,29057.585772
39,2022-09-02 15:00:00,24647.568577,26,33,5,15,Nublado,Nublado,28500.000000,30000.000000
40,2022-09-02 16:00:00,25500.000000,27,34,4,16,Nublado,Nublado,24647.568577,28062.328964
41,2022-09-02 17:00:00,24281.956494,28,36,2,17,Nublado,Nublado,25500.000000,28786.629243
42,2022-09-02 18:00:00,22733.515002,26,39,1,18,Nublado,Nublado,24281.956494,29900.303971
43,2022-09-02 19:00:00,11972.590689,25,44,1,19,Nublado,Nublado,22733.515002,18282.505369
61,2022-09-03 13:00:00,20400.000000,23,48,5,13,Nublado,Nublado,17723.695569,20596.278869
65,2022-09-03 17:00:00,25602.778606,25,45,5,17,Nublado,Nublado,23122.803757,24281.956494


In [4]:
columns = datos_dia.drop(columns=["Fecha", "Generación", "Cluster KMeans", "Cluster GMM"]).columns

In [5]:
X = datos_dia[columns]
X

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
36,21,57,4,12,17036.043251,29196.986647
37,23,45,5,13,27523.885172,25478.471342
38,24,37,6,14,20596.278869,29057.585772
39,26,33,5,15,28500.000000,30000.000000
40,27,34,4,16,24647.568577,28062.328964
...,...,...,...,...,...,...
18281,26,31,4,16,25562.000000,25385.000000
18282,26,32,2,17,25386.000000,22664.000000
18283,25,33,1,18,22872.000000,15736.000000
18284,23,38,0,19,15825.000000,1407.000000


In [6]:
y = datos_dia[['Generación']]
y

,Generación
36,27523.885172
37,20596.278869
38,28500.000000
39,24647.568577
40,25500.000000
...,...
18281,25386.000000
18282,22872.000000
18283,15825.000000
18284,1450.000000


Dividimos entrenamiento, validación y prueba

In [7]:
train_size = int(0.7 * len(X))
val_size = int(0.85 * len(X))

In [8]:
# Entrenamiento, validación y prueba, 75, 15 y 15
X_train, y_train =  X.iloc[:train_size, :], y.iloc[:train_size, :]
X_val, y_val = X.iloc[train_size:val_size, :], y.iloc[train_size:val_size, :]
X_test, y_test = X.iloc[val_size:, :],  y.iloc[val_size:,:]

print(f'X_train: {len(X_train)}, y_train: {len(y_train)}')
print(f'X_val: {len(X_val)}, y_val: {len(y_val)}')
print(f'X_test: {len(X_test)}, y_test: {len(y_test)}')

X_train: 3252, y_train: 3252
X_val: 697, y_val: 697
X_test: 697, y_test: 697


## Escalar con MinMaxScaler

In [9]:
from sklearn.preprocessing import MinMaxScaler

In [10]:
x_scaler = MinMaxScaler().fit(X_train)
x_scaler

MinMaxScaler()

In [11]:
X_train_scaled = x_scaler.transform(X_train)
print(X_train_scaled)
print(X_train_scaled.shape)

[[0.43333333 0.98113208 0.4        0.4        0.56786811 0.97323289]
 [0.5        0.75471698 0.5        0.46666667 0.91746284 0.84928238]
 [0.53333333 0.60377358 0.6        0.53333333 0.68654263 0.96858619]
 ...
 [0.4        0.67924528 0.         1.         0.         0.        ]
 [0.26666667 0.81132075 0.4        0.33333333 0.6396     0.67536667]
 [0.4        0.56603774 0.5        0.4        0.66776667 0.65666667]]
(3252, 6)


In [12]:
X_train_scaled_df = pd.DataFrame(X_train_scaled, index=X_train.index, columns=X_train.columns)
X_train_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
36,0.433333,0.981132,0.4,0.400000,0.567868,0.973233
37,0.500000,0.754717,0.5,0.466667,0.917463,0.849282
38,0.533333,0.603774,0.6,0.533333,0.686543,0.968586
39,0.600000,0.528302,0.5,0.600000,0.950000,1.000000
40,0.633333,0.547170,0.4,0.666667,0.821586,0.935411
...,...,...,...,...,...,...
12596,0.633333,0.226415,0.0,0.866667,0.319667,0.173900
12597,0.500000,0.396226,0.0,0.933333,0.026400,0.000000
12598,0.400000,0.679245,0.0,1.000000,0.000000,0.000000
12612,0.266667,0.811321,0.4,0.333333,0.639600,0.675367


In [13]:
X_val_scaled = x_scaler.transform(X_val)
print(X_val_scaled)
print(X_val_scaled.shape)

[[0.5        0.39622642 0.6        0.46666667 0.65776667 0.71383333]
 [0.56666667 0.30188679 0.5        0.53333333 0.63403333 0.62716667]
 [0.63333333 0.22641509 0.4        0.6        0.63616667 0.62633333]
 ...
 [0.4        0.49056604 0.3        0.2        0.37583333 0.7706    ]
 [0.53333333 0.30188679 0.5        0.26666667 0.74553333 0.859     ]
 [0.9        0.05660377 0.7        0.6        0.84236667 0.94003333]]
(697, 6)


In [14]:
X_val_scaled_df = pd.DataFrame(X_val_scaled, index=X_val.index, columns=X_val.columns)
X_val_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
12614,0.500000,0.396226,0.6,0.466667,0.657767,0.713833
12615,0.566667,0.301887,0.5,0.533333,0.634033,0.627167
12616,0.633333,0.226415,0.4,0.600000,0.636167,0.626333
12617,0.666667,0.150943,0.3,0.666667,0.655400,0.626067
12618,0.733333,0.113208,0.2,0.733333,0.737567,0.527167
...,...,...,...,...,...,...
14590,0.633333,0.490566,0.0,1.000000,0.009467,0.000000
14601,0.300000,0.716981,0.2,0.133333,0.044967,0.422800
14602,0.400000,0.490566,0.3,0.200000,0.375833,0.770600
14603,0.533333,0.301887,0.5,0.266667,0.745533,0.859000


In [15]:
X_test_scaled = x_scaler.transform(X_test)
print(X_test_scaled)
print(X_test_scaled.shape)

[[0.93333333 0.03773585 0.2        0.66666667 0.83383333 0.91813333]
 [0.96666667 0.03773585 0.2        0.73333333 0.72056667 0.88203333]
 [0.93333333 0.03773585 0.1        0.8        0.67996667 0.7186    ]
 ...
 [0.56666667 0.52830189 0.1        0.8        0.7624     0.52453333]
 [0.5        0.62264151 0.         0.86666667 0.5275     0.0469    ]
 [0.46666667 0.75471698 0.         0.93333333 0.04833333 0.        ]]
(697, 6)


In [16]:
X_test_scaled_df = pd.DataFrame(X_test_scaled, index=X_test.index, columns=X_test.columns)
X_test_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
14609,0.933333,0.037736,0.2,0.666667,0.833833,0.918133
14610,0.966667,0.037736,0.2,0.733333,0.720567,0.882033
14611,0.933333,0.037736,0.1,0.800000,0.679967,0.718600
14612,0.900000,0.037736,0.0,0.866667,0.574900,0.273700
14613,0.800000,0.075472,0.0,0.933333,0.218967,0.009467
...,...,...,...,...,...,...
18281,0.600000,0.490566,0.4,0.666667,0.852067,0.846167
18282,0.600000,0.509434,0.2,0.733333,0.846200,0.755467
18283,0.566667,0.528302,0.1,0.800000,0.762400,0.524533
18284,0.500000,0.622642,0.0,0.866667,0.527500,0.046900


In [17]:
x_scaller_all = MinMaxScaler().fit(X)
print(x_scaller_all)

MinMaxScaler()


In [18]:
X_scaled = x_scaller_all.transform(X)
print(X_scaled)
print(X_scaled.shape)

[[0.41935484 0.94736842 0.4        0.4        0.56786811 0.97323289]
 [0.48387097 0.73684211 0.5        0.46666667 0.91746284 0.84928238]
 [0.51612903 0.59649123 0.6        0.53333333 0.68654263 0.96858619]
 ...
 [0.5483871  0.52631579 0.1        0.8        0.7624     0.52453333]
 [0.48387097 0.61403509 0.         0.86666667 0.5275     0.0469    ]
 [0.4516129  0.73684211 0.         0.93333333 0.04833333 0.        ]]
(4646, 6)


In [19]:
X_scaled_df = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)
X_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
36,0.419355,0.947368,0.4,0.400000,0.567868,0.973233
37,0.483871,0.736842,0.5,0.466667,0.917463,0.849282
38,0.516129,0.596491,0.6,0.533333,0.686543,0.968586
39,0.580645,0.526316,0.5,0.600000,0.950000,1.000000
40,0.612903,0.543860,0.4,0.666667,0.821586,0.935411
...,...,...,...,...,...,...
18281,0.580645,0.491228,0.4,0.666667,0.852067,0.846167
18282,0.580645,0.508772,0.2,0.733333,0.846200,0.755467
18283,0.548387,0.526316,0.1,0.800000,0.762400,0.524533
18284,0.483871,0.614035,0.0,0.866667,0.527500,0.046900


In [20]:
y_scaler = MinMaxScaler().fit(y_train)
print(y_scaler)

MinMaxScaler()


In [21]:
y_train_scaled = y_scaler.transform(y_train)
print(y_train_scaled)
print(y_train_scaled.shape)

[[0.91746284]
 [0.68654263]
 [0.95      ]
 ...
 [0.        ]
 [0.66776667]
 [0.65776667]]
(3252, 1)


In [22]:
y_train_scaled_df = pd.DataFrame(y_train_scaled, index=y_train.index, columns=y_train.columns)
y_train_scaled_df

,Generación
36,0.917463
37,0.686543
38,0.950000
39,0.821586
40,0.850000
...,...
12596,0.026400
12597,0.000000
12598,0.000000
12612,0.667767


In [23]:
y_val_scaled = y_scaler.transform(y_val)
print(y_val_scaled)
print(y_val_scaled.shape)

[[6.34033333e-01]
 [6.36166667e-01]
 [6.55400000e-01]
 [7.37566667e-01]
 [5.70933333e-01]
 [3.33533333e-01]
 [2.61000000e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [6.70400000e-01]
 [6.63633333e-01]
 [6.37766667e-01]
 [6.27166667e-01]
 [6.26333333e-01]
 [6.40166667e-01]
 [6.02466667e-01]
 [3.65333333e-01]
 [3.65333333e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [8.34700000e-01]
 [8.30300000e-01]
 [7.99833333e-01]
 [7.87100000e-01]
 [7.82933333e-01]
 [8.26866667e-01]
 [7.83933333e-01]
 [4.66100000e-01]
 [4.78000000e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [8.34700000e-01]
 [8.40866667e-01]
 [8.04600000e-01]
 [7.83966667e-01]
 [7.82933333e-01]
 [8.25166667e-01]
 [7.76633333e-01]
 [4.66033333e-01]
 [6.56666667e-01]
 [6.38400000e-01]
 [6.27833333e-01]
 [6.29333333e-01]
 [6.26066667e-01]
 [6.02466667e-01]
 [3.65333333e-01]
 [3.47666667e-02]
 [6.34033333e-01]
 [6.27166667e-01]
 [6.33666667e-01]
 [6.26066667e-01]
 [6.02466667e-01]
 [3.65333333e-01]
 [3.28833333e-01]
 [3.33833333e-01]
 [3.163000

In [24]:
y_val_scaled_df = pd.DataFrame(y_val_scaled, index=y_val.index, columns=y_val.columns)
y_val_scaled_df

,Generación
12614,0.634033
12615,0.636167
12616,0.655400
12617,0.737567
12618,0.570933
...,...
14590,0.000000
14601,0.375833
14602,0.745533
14603,0.742033


In [25]:
y_test_scaled = y_scaler.transform(y_test)
print(y_test_scaled)
print(y_test_scaled.shape)

[[0.72056667]
 [0.67996667]
 [0.5749    ]
 [0.21896667]
 [0.00756667]
 [0.        ]
 [0.81536667]
 [0.90996667]
 [0.80986667]
 [0.76953333]
 [0.5749    ]
 [0.22023333]
 [0.0078    ]
 [0.        ]
 [0.81026667]
 [0.76496667]
 [0.7186    ]
 [0.2851    ]
 [0.0097    ]
 [0.        ]
 [0.80986667]
 [0.8513    ]
 [0.7186    ]
 [0.2862    ]
 [0.01033333]
 [0.        ]
 [0.83926667]
 [0.72      ]
 [0.68456667]
 [0.5749    ]
 [0.2982    ]
 [0.0105    ]
 [0.        ]
 [0.        ]
 [0.05056667]
 [0.39346667]
 [0.6523    ]
 [0.72796667]
 [0.74653333]
 [0.75363333]
 [0.7465    ]
 [0.7461    ]
 [0.7414    ]
 [0.72333333]
 [0.6804    ]
 [0.57563333]
 [0.22806667]
 [0.00756667]
 [0.        ]
 [0.57076667]
 [0.71923333]
 [0.74653333]
 [0.66036667]
 [0.65023333]
 [0.7461    ]
 [0.74116667]
 [0.6299    ]
 [0.59496667]
 [0.58286667]
 [0.22686667]
 [0.00763333]
 [0.        ]
 [0.4228    ]
 [0.6523    ]
 [0.8983    ]
 [0.6337    ]
 [0.51133333]
 [0.64676667]
 [0.24633333]
 [0.00973333]
 [0.        ]
 [0.81

In [26]:
y_test_scaled_df = pd.DataFrame(y_test_scaled, index=y_test.index, columns=y_test.columns)
y_test_scaled_df

,Generación
14609,0.720567
14610,0.679967
14611,0.574900
14612,0.218967
14613,0.007567
...,...
18281,0.846200
18282,0.762400
18283,0.527500
18284,0.048333


In [27]:
y_scaller_all = MinMaxScaler().fit(y)
print(y_scaller_all)

MinMaxScaler()


In [28]:
y_scaled = y_scaller_all.transform(y)
print(y_scaled)
print(y_scaled.shape)

[[0.91746284]
 [0.68654263]
 [0.95      ]
 ...
 [0.5275    ]
 [0.04833333]
 [0.        ]]
(4646, 1)


In [29]:
y_scaled_df = pd.DataFrame(y_scaled, index=y.index, columns=y.columns)
y_scaled_df

,Generación
36,0.917463
37,0.686543
38,0.950000
39,0.821586
40,0.850000
...,...
18281,0.846200
18282,0.762400
18283,0.527500
18284,0.048333


## Definición de modelos

### RandomForest

In [30]:
from lightgbm import LGBMRegressor
import optuna
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
import seaborn as sns
from sklearn.metrics import mean_absolute_percentage_error as mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error as mean_absolute_error
from sklearn.metrics import mean_squared_error as mean_squared_error
from sklearn.metrics import r2_score as r2_score

In [31]:
# Inicializar listas para métricas
LightGBM_model = LGBMRegressor(num_leaves=500, subsample= 0.10698460631792395, colsample_bytree= 0.7272836809565294, min_data_in_leaf= 85)
LightGBM_model.fit(X_train_scaled_df, y_train_scaled_df)
resultados = pd.DataFrame(index = y_test_scaled_df.index, columns=["LightGBM"])
#Ciclo diario de predicción
for i in range(len(X_test)):
    inicio = i * 1
    fin = inicio + 1

    X_test_seg = X_test_scaled_df.iloc[inicio:fin, :]
    y_test_seg = y_test_scaled_df.iloc[inicio:fin]

    if len(X_test_seg) < 1:
        break

    y_pred = LightGBM_model.predict(X_test_seg)
    y_pred = y_scaler.inverse_transform(y_pred.reshape(-1, 1))
    y_pred = np.clip(y_pred, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

    resultados.iloc[i, 0] = y_pred[0, 0]

[LightGBM] [Warning] min_data_in_leaf is set=85, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=85
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] min_data_in_leaf is set=85, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=85
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.051028 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 620
[LightGBM] [Info] Number of data points in the train set: 3252, number of used features: 6
[LightGBM] [Info] Start training from score 0.515719
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further split

In [32]:
resultados

,LightGBM
14609,24312.477855
14610,21363.704588
14611,16662.192043
14612,6329.374684
14613,374.941399
...,...
18281,26751.870947
18282,23370.96659
18283,15084.108971
18284,1894.938407


In [33]:
predicciones = y_test.copy()
predicciones

,Generación
14609,21617.0
14610,20399.0
14611,17247.0
14612,6569.0
14613,227.0
...,...
18281,25386.0
18282,22872.0
18283,15825.0
18284,1450.0


In [34]:
predicciones["LightGBM"] = resultados["LightGBM"]
predicciones

,Generación,LightGBM
14609,21617.0,24312.477855
14610,20399.0,21363.704588
14611,17247.0,16662.192043
14612,6569.0,6329.374684
14613,227.0,374.941399
...,...,...
18281,25386.0,26751.870947
18282,22872.0,23370.96659
18283,15825.0,15084.108971
18284,1450.0,1894.938407


In [35]:
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['LightGBM'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['LightGBM']):.4f}")

MAE: 1504.7843
RMSE: 2382.9166
R²: 0.9394


## Random Forest

In [36]:
from sklearn.ensemble import RandomForestRegressor

In [37]:
#Modelo LightGBM
RF_model = RandomForestRegressor(
    criterion="squared_error",
    random_state=0,
    n_estimators=400,
    min_impurity_decrease=0,
    max_depth=None,
    bootstrap=True
)
RF_model.fit(X_train_scaled_df, y_train_scaled_df)
# Inicializar listas para métricas
resultados = pd.DataFrame(index = y_test_scaled_df.index, columns=["Random Forest"])
#Ciclo diario de predicción
for i in range(len(X_test)):
    inicio = i * 1
    fin = inicio + 1

    X_test_seg = X_test_scaled_df.iloc[inicio:fin, :]
    y_test_seg = y_test_scaled_df.iloc[inicio:fin]

    if len(X_test_seg) < 1:
        break

    y_pred = RF_model.predict(X_test_seg)
    y_pred = y_scaler.inverse_transform(y_pred.reshape(-1, 1))
    y_pred = np.clip(y_pred, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

    resultados.iloc[i, 0] = y_pred[0, 0]

In [38]:
predicciones["Random Forest"] = resultados["Random Forest"]
predicciones

,Generación,LightGBM,Random Forest
14609,21617.0,24312.477855,25032.668096
14610,20399.0,21363.704588,20701.133338
14611,17247.0,16662.192043,15987.507201
14612,6569.0,6329.374684,7427.492492
14613,227.0,374.941399,349.495
...,...,...,...
18281,25386.0,26751.870947,26019.654367
18282,22872.0,23370.96659,22637.749887
18283,15825.0,15084.108971,16064.234452
18284,1450.0,1894.938407,2119.455433


## Preparación redes neuronales

In [39]:
import numpy as np
import pandas as pd

def create_sliding_window_with_index(data_X, data_y, lookback):
    X, y, indices = [], [], []
    
    # Asegurar que `data_y` tiene los mismos índices que `data_X`
    data_y = data_y.reindex(data_X.index)

    max_index = len(data_X) - lookback

    for i in range(max_index):
        X.append(data_X.iloc[i:i + lookback].values)  # Ventana de entrada
        
        # Obtener el índice correcto en `data_y`
        y_index = data_X.index[i + lookback]

        # Extraer el valor correspondiente de `data_y`
        if y_index in data_y.index:
            y_value = data_y.loc[y_index]
            if isinstance(y_value, pd.Series):  # Si devuelve una serie, extraer el valor
                y_value = y_value.iloc[0]
        else:
            y_value = np.nan  # Si no está, asignamos NaN

        y.append(y_value)
        indices.append(y_index)  # 🔹 Guardamos el índice original de `data_y`

    # Convertimos `X` en un array y `y` en DataFrame conservando sus índices originales
    X_array = np.array(X)
    y_df = pd.DataFrame(y, index=indices, columns=['y'])  # 🔹 Conservamos los índices originales

    return X_array, y_df


In [40]:
lookback = 48  # Puedes ajustar a 24, 72, etc.

# Aplicar la ventana deslizante a cada conjunto
X_train_windowed, y_train_windowed = create_sliding_window_with_index(X_train_scaled_df, y_train_scaled_df, lookback)
X_val_windowed, y_val_windowed = create_sliding_window_with_index(X_val_scaled_df, y_val_scaled_df, lookback)
X_test_windowed, y_test_windowed = create_sliding_window_with_index(X_test_scaled_df, y_test_scaled_df, lookback)


In [41]:
print(f'X_train: {X_train_windowed.shape}, y_train: {y_train_windowed.shape}')
print(f'X_val: {X_val_windowed.shape}, y_val: {y_val_windowed.shape}')
print(f'X_test: {X_test_windowed.shape}, y_test: {y_test_windowed.shape}')

X_train: (3204, 48, 6), y_train: (3204, 1)
X_val: (649, 48, 6), y_val: (649, 1)
X_test: (649, 48, 6), y_test: (649, 1)


## CTNET

In [42]:
import tensorflow as tf
from tensorflow.keras import layers

In [43]:
def compile_and_fit(model, xtrain=X_train_windowed, ytrain=y_train_windowed, learning_rate=0.0001):
    model.compile(loss=[tf.keras.losses.MeanSquaredError()],
                  optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  metrics=[tf.keras.metrics.RootMeanSquaredError(), tf.keras.metrics.MeanAbsolutePercentageError(), tf.keras.metrics.MeanAbsoluteError()])
    
    history = model.fit(xtrain, ytrain, epochs=50,
                        batch_size=512, validation_split=0.2, verbose=1)
    return history

def Loss(train_loss, valid_loss):
    plt.plot(train_loss)
    plt.plot(valid_loss)
    plt.rcParams["figure.figsize"] = (15, 3)
    plt.title('Model Losses')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train Loss', 'Validation Loss'], loc='upper left')
    plt.savefig('out/loss_plot.png')
    plt.show()

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    x = layers.LayerNormalization()(inputs)
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=128, kernel_size=2, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(norm_x, norm_x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    return norm_x

def build_model(input_shape, head_size, num_heads, ff_dim, num_transformer_blocks, mlp_units, dropout=0, mlp_dropout=0):
    inputs = tf.keras.Input(shape=input_shape)
    x = inputs
    
    for _ in range(num_transformer_blocks):
        enc_out = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)
    
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(enc_out, enc_out)
    res = x + enc_out
    x = layers.LayerNormalization(epsilon=1e-6)(res)
    x = layers.GlobalAveragePooling1D(data_format="channels_first")(x)
    x = layers.Dense(832, activation="relu")(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(mlp_dropout)(x)

    outputs = layers.Dense(1)(x)
    
    return tf.keras.Model(inputs, outputs)

In [44]:
CTNET = build_model((X_train_windowed.shape[1], X_train_windowed.shape[2]), head_size=4, num_heads=3, ff_dim=32, num_transformer_blocks=3, mlp_units=[256], mlp_dropout=0.3, dropout=0.2)

In [45]:
history = compile_and_fit(CTNET)

Epoch 1/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 153s 4s/step - loss: 0.3892 - mean_absolute_error: 0.5067 - mean_absolute_percentage_error: 186859.7031 - root_mean_squared_error: 0.6238 - val_loss: 0.3786 - val_mean_absolute_error: 0.5311 - val_mean_absolute_percentage_error: 793283.3125 - val_root_mean_squared_error: 0.6153
Epoch 2/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 44s 4s/step - loss: 0.3854 - mean_absolute_error: 0.5066 - mean_absolute_percentage_error: 1175555.0000 - root_mean_squared_error: 0.6207 - val_loss: 0.3702 - val_mean_absolute_error: 0.5254 - val_mean_absolute_percentage_error: 1703847.7500 - val_root_mean_squared_error: 0.6084
Epoch 3/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 40s 2s/step - loss: 0.3713 - mean_absolute_error: 0.4960 - mean_absolute_percentage_error: 2362575.7500 - root_mean_squared_error: 0.6093 - val_loss: 0.3607 - val_mean_absolute_error: 0.5189 - val_mean_absolute_percentage_error: 2747888.7500 - val_root_mean_squared_error: 0.6006
Epoch 4/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 31s 4s/step - los

In [46]:
CTNET_predictions = CTNET.predict(X_test_windowed)
CTNET_predictions

21/21 ━━━━━━━━━━━━━━━━━━━━ 21s 396ms/step


array([[0.5059599 ],
       [0.5029956 ],
       [0.50568545],
       [0.5104246 ],
       [0.51614064],
       [0.5208788 ],
       [0.52315134],
       [0.52240264],
       [0.52098244],
       [0.5197147 ],
       [0.5172834 ],
       [0.5167684 ],
       [0.5174764 ],
       [0.5167272 ],
       [0.51516336],
       [0.51471823],
       [0.51799417],
       [0.52281886],
       [0.52508926],
       [0.5256483 ],
       [0.52525645],
       [0.5234683 ],
       [0.51955193],
       [0.5153869 ],
       [0.5203881 ],
       [0.52913827],
       [0.53392637],
       [0.5343628 ],
       [0.53545225],
       [0.53730553],
       [0.5409675 ],
       [0.54144615],
       [0.540031  ],
       [0.53888047],
       [0.54228854],
       [0.54683673],
       [0.547083  ],
       [0.5448188 ],
       [0.5426137 ],
       [0.53856796],
       [0.5339338 ],
       [0.5287147 ],
       [0.52364635],
       [0.52968425],
       [0.5299931 ],
       [0.5279184 ],
       [0.52630085],
       [0.527

In [47]:
CTNET_predictions = y_scaler.inverse_transform(CTNET_predictions.reshape(-1, 1))
CTNET_predictions = np.clip(CTNET_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [48]:
resultados = pd.DataFrame(CTNET_predictions, index = y_test_windowed.index, columns=["CTNET"])

In [49]:
predicciones["CTNET"] = resultados["CTNET"]
predicciones

,Generación,LightGBM,Random Forest,CTNET
14609,21617.0,24312.477855,25032.668096,NaN
14610,20399.0,21363.704588,20701.133338,NaN
14611,17247.0,16662.192043,15987.507201,NaN
14612,6569.0,6329.374684,7427.492492,NaN
14613,227.0,374.941399,349.495,NaN
...,...,...,...,...
18281,25386.0,26751.870947,26019.654367,17638.589844
18282,22872.0,23370.96659,22637.749887,17667.765625
18283,15825.0,15084.108971,16064.234452,17651.998047
18284,1450.0,1894.938407,2119.455433,17606.646484


In [50]:
predicciones["CTNET"] = predicciones["CTNET"].fillna(0)

In [51]:
# import optuna
# import tensorflow as tf
# from tensorflow.keras import layers
# from sklearn.model_selection import train_test_split

# # Definir la función objetivo para Optuna
# def objective(trial):
#     # Sugerir valores para los hiperparámetros
#     head_size = trial.suggest_int("head_size", 8, 64, step=8)
#     num_heads = trial.suggest_int("num_heads", 2, 8, step=2)
#     ff_dim = trial.suggest_int("ff_dim", 32, 256, step=32)
#     num_transformer_blocks = trial.suggest_int("num_transformer_blocks", 1, 4)
#     mlp_units = trial.suggest_categorical("mlp_units", [[128, 64], [256, 128, 64], [512, 256, 128]])
#     dropout = trial.suggest_float("dropout", 0.1, 0.5, step=0.1)
#     mlp_dropout = trial.suggest_float("mlp_dropout", 0.1, 0.5, step=0.1)
#     learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 1e-2)

#     # Construcción del modelo con los hiperparámetros sugeridos
#     model = build_model(
#         input_shape=X_train_windowed.shape[1:],
#         head_size=head_size,
#         num_heads=num_heads,
#         ff_dim=ff_dim,
#         num_transformer_blocks=num_transformer_blocks,
#         mlp_units=mlp_units,
#         dropout=dropout,
#         mlp_dropout=mlp_dropout
#     )

#     # Compilar el modelo con los hiperparámetros sugeridos
#     model.compile(
#         loss=tf.keras.losses.MeanSquaredError(),
#         optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
#         metrics=[tf.keras.metrics.RootMeanSquaredError()]
#     )

#     # Entrenamiento con un número reducido de épocas para acelerar la búsqueda
#     history = model.fit(
#         X_train_windowed, y_train_windowed,
#         validation_split=0.2,
#         epochs=50,  # Reducimos las épocas para acelerar la búsqueda
#         batch_size=512,
#         verbose=0
#     )

#     # Obtener la métrica de validación (RMSE) y minimizarla
#     val_rmse = min(history.history["val_root_mean_squared_error"])
    
#     return val_rmse  # Queremos minimizar el RMSE

# # Ejecutar la optimización de hiperparámetros
# study = optuna.create_study(direction="minimize")
# study.optimize(objective, n_trials=20, timeout=3600)  # 20 iteraciones, máximo 1 hora

# # Mostrar los mejores hiperparámetros encontrados
# best_params = study.best_params
# print(f"Mejores hiperparámetros: {best_params}")


In [52]:
CTNET = build_model((X_train_windowed.shape[1], X_train_windowed.shape[2]), head_size=16, num_heads=8, ff_dim=256, num_transformer_blocks=1, mlp_units=[128,64], mlp_dropout=0.2, dropout=0.2)

In [53]:
history = compile_and_fit(CTNET, learning_rate = 0.0010762230908145116)

Epoch 1/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 219s 10s/step - loss: 0.3658 - mean_absolute_error: 0.4951 - mean_absolute_percentage_error: 4155668.5000 - root_mean_squared_error: 0.6046 - val_loss: 0.1970 - val_mean_absolute_error: 0.3951 - val_mean_absolute_percentage_error: 25299122.0000 - val_root_mean_squared_error: 0.4439
Epoch 2/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 77s 8s/step - loss: 0.1800 - mean_absolute_error: 0.3645 - mean_absolute_percentage_error: 48039816.0000 - root_mean_squared_error: 0.4237 - val_loss: 0.1719 - val_mean_absolute_error: 0.3170 - val_mean_absolute_percentage_error: 92058688.0000 - val_root_mean_squared_error: 0.4146
Epoch 3/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 73s 6s/step - loss: 0.2127 - mean_absolute_error: 0.3680 - mean_absolute_percentage_error: 107094784.0000 - root_mean_squared_error: 0.4610 - val_loss: 0.1055 - val_mean_absolute_error: 0.2834 - val_mean_absolute_percentage_error: 51195656.0000 - val_root_mean_squared_error: 0.3249
Epoch 4/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 49s 7s/s

In [54]:
CTNET_predictions = CTNET.predict(X_test_windowed)
CTNET_predictions

21/21 ━━━━━━━━━━━━━━━━━━━━ 21s 423ms/step


array([[0.677957  ],
       [0.74930465],
       [0.9543879 ],
       [0.99708134],
       [0.9356892 ],
       [0.7655897 ],
       [0.758396  ],
       [0.67938036],
       [0.38771638],
       [0.31287763],
       [0.28373677],
       [0.43926272],
       [0.6370201 ],
       [0.6657094 ],
       [0.7030028 ],
       [0.7899409 ],
       [1.0917351 ],
       [0.9970025 ],
       [0.5630158 ],
       [0.49899086],
       [0.49302605],
       [0.57160234],
       [0.5249633 ],
       [0.5066996 ],
       [0.7568704 ],
       [0.82791495],
       [0.85321474],
       [0.8378831 ],
       [0.57359695],
       [0.47416192],
       [0.5543943 ],
       [0.6721227 ],
       [0.6556852 ],
       [0.600535  ],
       [0.8069048 ],
       [0.87511694],
       [0.835178  ],
       [0.5116378 ],
       [0.42611554],
       [0.31728292],
       [0.41420585],
       [0.5744175 ],
       [0.5831064 ],
       [0.8101547 ],
       [0.7289185 ],
       [0.69380796],
       [0.5078384 ],
       [0.584

In [55]:
CTNET_predictions = y_scaler.inverse_transform(CTNET_predictions.reshape(-1, 1))
CTNET_predictions = np.clip(CTNET_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [56]:
resultados = pd.DataFrame(CTNET_predictions, index = y_test_windowed.index, columns=["CTNET"])

In [57]:
predicciones["CTNET"] = resultados["CTNET"]
predicciones

,Generación,LightGBM,Random Forest,CTNET
14609,21617.0,24312.477855,25032.668096,NaN
14610,20399.0,21363.704588,20701.133338,NaN
14611,17247.0,16662.192043,15987.507201,NaN
14612,6569.0,6329.374684,7427.492492,NaN
14613,227.0,374.941399,349.495,NaN
...,...,...,...,...
18281,25386.0,26751.870947,26019.654367,26938.273438
18282,22872.0,23370.96659,22637.749887,19476.591797
18283,15825.0,15084.108971,16064.234452,14291.337891
18284,1450.0,1894.938407,2119.455433,12272.419922


## Forecasting

In [58]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.metrics import RootMeanSquaredError
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.losses import Huber
from tensorflow.keras.callbacks import EarlyStopping

In [59]:
Forecast_model = Sequential()
Forecast_model.add(InputLayer((X_train_windowed.shape[1], X_train_windowed.shape[2])))

#CNN
Forecast_model.add(Conv1D(filters=64, kernel_size=2, padding='same', activation='relu'))
Forecast_model.add(BatchNormalization())  # 🔹 Nueva Normalización aquí
Forecast_model.add(MaxPooling1D(pool_size=2))

#model_Soleado.add(Flatten())
#BiLSTM
Forecast_model.add(Bidirectional(LSTM(128, return_sequences=True)))
Forecast_model.add(Bidirectional(LSTM(64, return_sequences=True)))
Forecast_model.add(Dropout(0.2))  # 🔹 Mayor regularización en BiLSTM
Forecast_model.add(Bidirectional(LSTM(32, return_sequences=False)))

#Normalización y Dropout
Forecast_model.add(BatchNormalization())
Forecast_model.add(Dropout(0.3))

# Capas Densas
Forecast_model.add(Dense(16, activation='relu'))
Forecast_model.add(Dense(1, 'relu'))

Forecast_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_12 (Conv1D)              │ (None, 48, 64)         │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 48, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 24, 256)        │       197,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 24, 128)        │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 24, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 16)             │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 405,601 (1.55 MB)

 Trainable params: 405,345 (1.55 MB)

 Non-trainable params: 256 (1.00 KB)

In [60]:
cp = ModelCheckpoint('Forcasting_model.keras', save_best_only=True)
Forecast_model.compile(optimizer=Adam(learning_rate=0.0001), loss=Huber(delta=1000), metrics=['mae'])
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [61]:
history = Forecast_model.fit(X_train_windowed, y_train_windowed, validation_data=(X_val_windowed, y_val_windowed), epochs=100, batch_size=8, callbacks=[cp, early_stop])

Epoch 1/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 549s 560ms/step - loss: 0.1648 - mae: 0.4687 - val_loss: 0.0989 - val_mae: 0.3939
Epoch 2/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 215s 523ms/step - loss: 0.1131 - mae: 0.3854 - val_loss: 0.0422 - val_mae: 0.2320
Epoch 3/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 269s 529ms/step - loss: 0.0902 - mae: 0.3417 - val_loss: 0.0293 - val_mae: 0.2017
Epoch 4/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 262s 514ms/step - loss: 0.0795 - mae: 0.3201 - val_loss: 0.0236 - val_mae: 0.1771
Epoch 5/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 275s 533ms/step - loss: 0.0709 - mae: 0.3021 - val_loss: 0.0193 - val_mae: 0.1556
Epoch 6/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 259s 509ms/step - loss: 0.0619 - mae: 0.2743 - val_loss: 0.0170 - val_mae: 0.1400
Epoch 7/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 247s 461ms/step - loss: 0.0565 - mae: 0.2571 - val_loss: 0.0160 - val_mae: 0.1335
Epoch 8/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 192s 426ms/step - loss: 0.0548 - mae: 0.2547 - val_loss: 0.0154 - val_mae: 0.1225
Epoch 9/100
401/

In [62]:
Forecast_predictions = Forecast_model.predict(X_test_windowed)
Forecast_predictions

21/21 ━━━━━━━━━━━━━━━━━━━━ 70s 2s/step 


array([[0.        ],
       [0.        ],
       [0.77839005],
       [0.7316328 ],
       [0.75527555],
       [0.8038016 ],
       [0.76146895],
       [0.7211652 ],
       [0.64035994],
       [0.37215838],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.57861173],
       [0.62578595],
       [0.69638985],
       [0.6923325 ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.7432423 ],
       [0.78991807],
       [0.80969614],
       [0.8199906 ],
       [0.6786061 ],
       [0.52533215],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.6226164 ],
       [0.8046098 ],
       [0.7970454 ],
       [0.82858855],
       [0.80587286],
       [0.7183381 ],
       [0.4957407 ],
       [0.        ],
       [0.27093333],
       [0.7703099 ],
       [0.8251013 ],
       [0.75843847],
       [0.6297493 ],
       [0.        ],
       [0.   

In [63]:
Forecast_predictions = y_scaler.inverse_transform(Forecast_predictions.reshape(-1, 1))
Forecast_predictions = np.clip(Forecast_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [64]:
Forecast_resultados = pd.DataFrame(Forecast_predictions, index = y_test_windowed.index, columns=["Forecast"])

In [65]:
predicciones["Forecast"] = Forecast_resultados["Forecast"]
predicciones

,Generación,LightGBM,Random Forest,CTNET,Forecast
14609,21617.0,24312.477855,25032.668096,NaN,NaN
14610,20399.0,21363.704588,20701.133338,NaN,NaN
14611,17247.0,16662.192043,15987.507201,NaN,NaN
14612,6569.0,6329.374684,7427.492492,NaN,NaN
14613,227.0,374.941399,349.495,NaN,NaN
...,...,...,...,...,...
18281,25386.0,26751.870947,26019.654367,26938.273438,23562.326172
18282,22872.0,23370.96659,22637.749887,19476.591797,22858.564453
18283,15825.0,15084.108971,16064.234452,14291.337891,18372.794922
18284,1450.0,1894.938407,2119.455433,12272.419922,6532.208008


## Métricas

In [66]:
predicciones.loc[~predicciones['CTNET'].isna(),'Generación']

14734        0.0
14746    17123.0
14747    21577.0
14748    22396.0
14749    19811.0
          ...   
18281    25386.0
18282    22872.0
18283    15825.0
18284     1450.0
18285        0.0
Name: Generación, Length: 649, dtype: float64

## Photovoltaic

In [67]:
from tensorflow.keras.models import Model
inputs = Input(shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]))

# Primera capa CNN
x = Conv1D(filters=64, kernel_size=4, padding='same', activation='relu')(inputs)
x = MaxPooling1D(pool_size=2)(x)

# Segunda capa CNN
x = Conv1D(filters=128, kernel_size=4, padding='same', activation='relu')(x)
x = MaxPooling1D(pool_size=2)(x)

# Capa BiGRU
x = Bidirectional(GRU(64, return_sequences=True))(x)

# Atención: se define de forma explícita
attention = MultiHeadAttention(num_heads=4, key_dim=128)(x, x)

# Aplanar y agregar Dropout
x = Flatten()(attention)
x = Dropout(0.4)(x)
initializer = tf.keras.initializers.HeNormal()
x = Dense(64, activation="relu", kernel_regularizer=l2(0.01))(x)
x = Dense(32, activation="relu")(x)  # Otra capa intermedia

# Capa de salida
outputs = Dense(1, activation="linear")(x)

# Definir el modelo
Photo_model = Model(inputs=inputs, outputs=outputs)

# Resumen del modelo
Photo_model.summary()

Model: "functional_13"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 48, 6)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_13 (Conv1D)  │ (None, 48, 64)    │      1,600 │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 24, 64)    │          0 │ conv1d_13[0][0]   │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_14 (Conv1D)  │ (None, 24, 128)   │     32,896 │ max_pooling1d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_2     │ (None, 12, 128)   │          0 │ conv1d_14[0][0]   │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_3     │ (None, 12, 128)   │     74,496 │ max_pooling1d_2[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 12, 128)   │    263,808 │ bidirectional_3[… │
│ (MultiHeadAttentio… │                   │            │ bidirectional_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 1536)      │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_11          │ (None, 1536)      │          0 │ flatten[0][0]     │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 64)        │     98,368 │ dropout_11[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 32)        │      2,080 │ dense_10[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 1)         │         33 │ dense_11[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 473,281 (1.81 MB)

 Trainable params: 473,281 (1.81 MB)

 Non-trainable params: 0 (0.00 B)

In [68]:
cp2 = ModelCheckpoint('Photovoltaic_model.keras', save_best_only=True)
Photo_model.compile(optimizer=Adam(learning_rate=0.0001), loss="mean_squared_error", metrics=['mae'])
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [69]:
history = Photo_model.fit(
    X_train_windowed, y_train_windowed,
    validation_data=(X_val_windowed, y_val_windowed),
    epochs=50,
    batch_size=16,
    callbacks=[cp, early_stop]
)

Epoch 1/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 313s 288ms/step - loss: 1.1948 - mae: 0.3311 - val_loss: 0.7217 - val_mae: 0.3413
Epoch 2/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 52s 122ms/step - loss: 0.6148 - mae: 0.3000 - val_loss: 0.3851 - val_mae: 0.2907
Epoch 3/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 61s 208ms/step - loss: 0.3141 - mae: 0.2214 - val_loss: 0.1768 - val_mae: 0.1431
Epoch 4/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 53s 231ms/step - loss: 0.1675 - mae: 0.1585 - val_loss: 0.1076 - val_mae: 0.1308
Epoch 5/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 77s 189ms/step - loss: 0.1040 - mae: 0.1397 - val_loss: 0.0737 - val_mae: 0.1271
Epoch 6/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 38s 155ms/step - loss: 0.0781 - mae: 0.1462 - val_loss: 0.0541 - val_mae: 0.1182
Epoch 7/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 55s 205ms/step - loss: 0.0619 - mae: 0.1362 - val_loss: 0.0458 - val_mae: 0.1182
Epoch 8/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 94s 241ms/step - loss: 0.0495 - mae: 0.1263 - val_loss: 0.0376 - val_mae: 0.1111
Epoch 9/50
201/201 ━━━━━━━━━━━━

In [70]:
Photo_predictions = Photo_model.predict(X_test_windowed)
Photo_predictions

21/21 ━━━━━━━━━━━━━━━━━━━━ 28s 685ms/step


array([[0.00851196],
       [0.429486  ],
       [0.7655398 ],
       [0.74209803],
       [0.7376799 ],
       [0.7590738 ],
       [0.7907348 ],
       [0.8023163 ],
       [0.7716597 ],
       [0.67604625],
       [0.47289535],
       [0.16891873],
       [0.00713791],
       [0.00846365],
       [0.29486433],
       [0.57771295],
       [0.65203995],
       [0.713242  ],
       [0.73109394],
       [0.66442245],
       [0.34734085],
       [0.03475136],
       [0.01746897],
       [0.37449226],
       [0.67329645],
       [0.7451466 ],
       [0.7475993 ],
       [0.77037007],
       [0.7185883 ],
       [0.63711405],
       [0.593696  ],
       [0.09774473],
       [0.00961406],
       [0.47990647],
       [0.75913084],
       [0.8065916 ],
       [0.78937525],
       [0.7334015 ],
       [0.6701183 ],
       [0.45070025],
       [0.02541204],
       [0.3537065 ],
       [0.71692103],
       [0.7793214 ],
       [0.7750149 ],
       [0.65354806],
       [0.09210671],
       [0.010

In [71]:
Photo_predictions = y_scaler.inverse_transform(Photo_predictions.reshape(-1, 1))
Photo_predictions = np.clip(Photo_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [72]:
Photo_resultados = pd.DataFrame(Photo_predictions, index = y_test_windowed.index, columns=["Photo"])

In [73]:
predicciones["Photo"] = Photo_resultados["Photo"]
predicciones

,Generación,LightGBM,Random Forest,CTNET,Forecast,Photo
14609,21617.0,24312.477855,25032.668096,NaN,NaN,NaN
14610,20399.0,21363.704588,20701.133338,NaN,NaN,NaN
14611,17247.0,16662.192043,15987.507201,NaN,NaN,NaN
14612,6569.0,6329.374684,7427.492492,NaN,NaN,NaN
14613,227.0,374.941399,349.495,NaN,NaN,NaN
...,...,...,...,...,...,...
18281,25386.0,26751.870947,26019.654367,26938.273438,23562.326172,25576.613281
18282,22872.0,23370.96659,22637.749887,19476.591797,22858.564453,22927.134766
18283,15825.0,15084.108971,16064.234452,14291.337891,18372.794922,15126.110352
18284,1450.0,1894.938407,2119.455433,12272.419922,6532.208008,4951.829102


In [74]:
print("LightGBM")
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['LightGBM'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print("Random Forest")
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['Random Forest']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['Random Forest'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['Random Forest']):.4f}")
print("CTNET")
print(f"MAE: {mean_absolute_error(predicciones.loc[~predicciones['CTNET'].isna(),'Generación'], predicciones.loc[~predicciones['CTNET'].isna(),'CTNET']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones.loc[~predicciones['CTNET'].isna(),'Generación'], predicciones.loc[~predicciones['CTNET'].isna(),'CTNET'])):.4f}")
print(f"R²: {r2_score(predicciones.loc[~predicciones['CTNET'].isna(),'Generación'], predicciones.loc[~predicciones['CTNET'].isna(),'CTNET']):.4f}")
print("Forecast")
print(f"MAE: {mean_absolute_error(predicciones.loc[~predicciones['Forecast'].isna(),'Generación'], predicciones.loc[~predicciones['Forecast'].isna(),'Forecast']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones.loc[~predicciones['Forecast'].isna(),'Generación'], predicciones.loc[~predicciones['Forecast'].isna(),'Forecast'])):.4f}")
print(f"R²: {r2_score(predicciones.loc[~predicciones['Forecast'].isna(),'Generación'], predicciones.loc[~predicciones['Forecast'].isna(),'Forecast']):.4f}")
print("Photovoltaic")
print(f"MAE: {mean_absolute_error(predicciones.loc[~predicciones['Photo'].isna(),'Generación'], predicciones.loc[~predicciones['Photo'].isna(),'Photo']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones.loc[~predicciones['Photo'].isna(),'Generación'], predicciones.loc[~predicciones['Photo'].isna(),'Photo'])):.4f}")
print(f"R²: {r2_score(predicciones.loc[~predicciones['Photo'].isna(),'Generación'], predicciones.loc[~predicciones['Photo'].isna(),'Photo']):.4f}")

LightGBM
MAE: 1504.7843
RMSE: 2382.9166
R²: 0.9394
Random Forest
MAE: 1513.2853
RMSE: 2476.0689
R²: 0.9345
CTNET
MAE: 6632.2267
RMSE: 8413.6818
R²: 0.2375
Forecast
MAE: 4183.7473
RMSE: 5805.3628
R²: 0.6370
Photovoltaic
MAE: 3708.5321
RMSE: 5241.4415
R²: 0.7041


In [75]:
# Seleccionar las columnas desde "LightGBM" en adelante
columnas_nuevas = predicciones.loc[:, "LightGBM":]

# Unir con `datos` usando el índice, manteniendo todo en `datos`
datos = datos.merge(columnas_nuevas, left_index=True, right_index=True, how='left')

# Ver resultado
datos.head()


,Fecha,Generación,Temperatura,Humedad Relativa,Índice UV,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day,LightGBM,Random Forest,CTNET,Forecast,Photo
24,2022-09-02 00:00:00,0.0,19,76,0,0,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
25,2022-09-02 01:00:00,0.0,18,81,0,1,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
26,2022-09-02 02:00:00,0.0,18,84,0,2,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
27,2022-09-02 03:00:00,0.0,18,86,0,3,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
28,2022-09-02 04:00:00,0.0,17,86,0,4,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN


## X_train para hacer análisis de sobreajuste

In [76]:
predicciones_train = y_train.copy()

In [77]:
LightGBM_predictions_train = LightGBM_model.predict(X_train_scaled_df)
LightGBM_predictions_train = y_scaler.inverse_transform(LightGBM_predictions_train.reshape(-1, 1))
LightGBM_predictions_train = np.clip(LightGBM_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
LightGBM_resultados = pd.DataFrame(LightGBM_predictions_train, index = y_train_scaled_df.index, columns=["LightGBM_train"])
predicciones_train["LightGBM_train"] = LightGBM_resultados["LightGBM_train"]

[LightGBM] [Warning] min_data_in_leaf is set=85, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=85


In [78]:
RandomForest_predictions_train = RF_model.predict(X_train_scaled_df)
RandomForest_predictions_train = y_scaler.inverse_transform(RandomForest_predictions_train.reshape(-1, 1))
RandomForest_predictions_train = np.clip(RandomForest_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
RandomForest_resultados = pd.DataFrame(RandomForest_predictions_train, index = y_train_scaled_df.index, columns=["RandomForest_train"])
predicciones_train["RandomForest_train"] = RandomForest_resultados["RandomForest_train"]

In [79]:
CTNET_predictions_train = CTNET.predict(X_train_windowed)
CTNET_predictions_train = y_scaler.inverse_transform(CTNET_predictions_train.reshape(-1, 1))
CTNET_predictions_train = np.clip(CTNET_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
CTNET_resultados = pd.DataFrame(CTNET_predictions_train, index = y_train_windowed.index, columns=["CTNET_train"])
predicciones_train["CTNET_train"] = CTNET_resultados["CTNET_train"]

101/101 ━━━━━━━━━━━━━━━━━━━━ 15s 93ms/step


In [80]:
Forecast_predictions_train = Forecast_model.predict(X_train_windowed)
Forecast_predictions_train = y_scaler.inverse_transform(Forecast_predictions_train.reshape(-1, 1))
Forecast_predictions_train = np.clip(Forecast_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
Forecast_resultados = pd.DataFrame(Forecast_predictions_train, index = y_train_windowed.index, columns=["Forecast_train"])
predicciones_train["Forecast_train"] = Forecast_resultados["Forecast_train"]

101/101 ━━━━━━━━━━━━━━━━━━━━ 15s 94ms/step


In [81]:
Photo_predictions_train = Photo_model.predict(X_train_windowed)
Photo_predictions_train = y_scaler.inverse_transform(Photo_predictions_train.reshape(-1, 1))
Photo_predictions_train = np.clip(Photo_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
Photo_resultados = pd.DataFrame(Photo_predictions_train, index = y_train_windowed.index, columns=["Photo_train"])
predicciones_train["Photo_train"] = Photo_resultados["Photo_train"]

101/101 ━━━━━━━━━━━━━━━━━━━━ 6s 36ms/step


In [82]:
predicciones_train

,Generación,LightGBM_train,RandomForest_train,CTNET_train,Forecast_train,Photo_train
36,27523.885172,21365.227112,23981.312960,NaN,NaN,NaN
37,20596.278869,26648.355787,23474.697202,NaN,NaN,NaN
38,28500.000000,22480.825478,25083.710825,NaN,NaN,NaN
39,24647.568577,27450.230111,25678.095627,NaN,NaN,NaN
40,25500.000000,23918.077465,25129.541912,NaN,NaN,NaN
...,...,...,...,...,...,...
12596,792.000000,2535.317889,2228.205000,12075.732422,0.000000,1547.947144
12597,0.000000,0.000000,0.000000,9251.398438,0.000000,102.846786
12598,0.000000,95.412834,0.000000,16431.587891,0.000000,383.516602
12612,20033.000000,21501.104937,19819.580000,23643.132812,24218.046875,23807.849609


In [83]:
print("LightGBM")
print(f"MAE: {mean_absolute_error(predicciones_train['Generación'], predicciones_train['LightGBM_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train['Generación'], predicciones_train['LightGBM_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train['Generación'], predicciones_train['LightGBM_train']):.4f}")
print("Random Forest")
print(f"MAE: {mean_absolute_error(predicciones_train['Generación'], predicciones_train['RandomForest_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train['Generación'], predicciones_train['RandomForest_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train['Generación'], predicciones_train['RandomForest_train']):.4f}")
print("CTNET")
print(f"MAE: {mean_absolute_error(predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'CTNET_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'CTNET_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'CTNET_train']):.4f}")
print("Forecast")
print(f"MAE: {mean_absolute_error(predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Forecast_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Forecast_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Forecast_train']):.4f}")
print("Photovoltaic")
print(f"MAE: {mean_absolute_error(predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Photo_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Photo_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Photo_train']):.4f}")

LightGBM
MAE: 1055.3486
RMSE: 1926.5182
R²: 0.9670
Random Forest
MAE: 379.4911
RMSE: 732.1718
R²: 0.9952
CTNET
MAE: 6804.4540
RMSE: 8648.7357
R²: 0.3370
Forecast
MAE: 4186.9731
RMSE: 6044.1702
R²: 0.6762
Photovoltaic
MAE: 2798.8321
RMSE: 4274.4204
R²: 0.8381


In [84]:
# Seleccionar las columnas desde "LightGBM" en adelante
columnas_nuevas = predicciones_train.loc[:, "LightGBM_train":]

# Unir con `datos` usando el índice, manteniendo todo en `datos`
datos = datos.merge(columnas_nuevas, left_index=True, right_index=True, how='left')

# Ver resultado
datos.head()


,Fecha,Generación,Temperatura,Humedad Relativa,Índice UV,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day,LightGBM,Random Forest,CTNET,Forecast,Photo,LightGBM_train,RandomForest_train,CTNET_train,Forecast_train,Photo_train
24,2022-09-02 00:00:00,0.0,19,76,0,0,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25,2022-09-02 01:00:00,0.0,18,81,0,1,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26,2022-09-02 02:00:00,0.0,18,84,0,2,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2022-09-02 03:00:00,0.0,18,86,0,3,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28,2022-09-02 04:00:00,0.0,17,86,0,4,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [85]:
datos.to_excel("04.11_Predicciones_Conjunto_nublado GMM CTNET.xlsx", index=True)

## Guardamos los modelos

In [86]:
import joblib

# Guardar modelo LightGBM
joblib.dump(LightGBM_model, "4_11_LightGBM_model.pkl")

# Guardar modelo Random Forest
joblib.dump(RF_model, "4_11_RandomForest_model.pkl")


['4_11_RandomForest_model.pkl']

In [87]:
CTNET.save("4_11_CTNET_model.keras")
Forecast_model.save("4_11_Forecast_model.keras")
Photo_model.save("4_11_Photo_model.keras")